# Fast Custom CNN + Live Product Detection (17 categories)

This notebook trains a **MobileNetV2 transfer-learning CNN classifier** from your own product photos and uses **YOLO-World** to propose real object boxes in the live webcam. The CNN labels each detected crop; boxes update as the camera view changes. No Supabase is used.

**Important:** 5 images per category is only a tiny starter dataset. Augmentation helps but cannot create new real-world variety. The notebook does not claim guaranteed accuracy. For stronger detection of every category, annotate and train a custom YOLO detector with bounding boxes. YOLO-World may miss unusual objects such as wires, erasers, or toothpaste; the CNN can only classify objects that have been cropped/detected.

Run cells from top to bottom. Select a GPU in **Runtime → Change runtime type → T4 GPU** if available.

In [ ]:
!pip -q install tensorflow==2.16.1 ultralytics gradio pillow scikit-learn ddgs requests


## 1. Automatically build a starter dataset from online images

You do **not** need to upload a dataset. The next cell searches public image-search results for each category, downloads usable images, and stores them in class folders automatically. Search results can include incorrect or copyrighted images; this is a quick prototype dataset, not a curated benchmark. Review/replace poor images if you want better accuracy.

The notebook attempts to download 20 images per category. Search engines can rate-limit or block requests, so some categories may end up with fewer images.


In [ ]:
import os, time, requests, pathlib, random, io
from PIL import Image, ImageFile
from ddgs import DDGS
ImageFile.LOAD_TRUNCATED_IMAGES = True

CLASSES = ['biscuit','chips','pen','scale','phone','calculator','headphone','laptop','books','pencils','erasers','keyboard','mouse','toothpaste','ball','toothbrush','wires']
SEARCH_TERMS = {
 'biscuit':'biscuit packet packaged biscuits product photo',
 'chips':'potato chips packet snack packet product photo',
 'pen':'ballpoint pen stationery product photo',
 'scale':'ruler measuring scale stationery product photo',
 'phone':'smartphone mobile phone product photo',
 'calculator':'handheld calculator product photo',
 'headphone':'headphones headset product photo',
 'laptop':'laptop computer product photo',
 'books':'school books notebooks stack product photo',
 'pencils':'pencils stationery product photo',
 'erasers':'rubber eraser stationery product photo',
 'keyboard':'computer keyboard product photo',
 'mouse':'computer mouse product photo',
 'toothpaste':'toothpaste tube product photo',
 'ball':'sports ball product photo',
 'toothbrush':'toothbrush product photo',
 'wires':'electrical wires cables product photo'
}
DATA_DIR='/content/product_images'
os.makedirs(DATA_DIR,exist_ok=True)
TARGET_PER_CLASS=20  # increase to 30-50 later for a better starter dataset
session=requests.Session()
session.headers.update({'User-Agent':'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124 Safari/537.36'})
def download_class(class_name, query, target=TARGET_PER_CLASS):
    folder=os.path.join(DATA_DIR,class_name); os.makedirs(folder,exist_ok=True)
    existing=list(pathlib.Path(folder).glob('*.jpg'))
    if len(existing)>=target: return len(existing)
    try:
        results=DDGS().images(query, region='us-en', safesearch='moderate', max_results=target*3)
    except Exception as e:
        print(f'Image search failed for {class_name}: {e}'); return len(existing)
    saved=len(existing); seen=set()
    for item in results:
        if saved>=target: break
        url=item.get('image') or item.get('thumbnail')
        if not url or url in seen: continue
        seen.add(url)
        try:
            response=session.get(url,timeout=10); response.raise_for_status()
            im=Image.open(io.BytesIO(response.content)).convert('RGB')
            if im.width<80 or im.height<80: continue
            im.thumbnail((900,900))
            out=os.path.join(folder,f'{class_name}_{saved+1:03d}.jpg')
            im.save(out,'JPEG',quality=90)
            saved+=1
        except Exception:
            continue
    return saved

counts={}
for cls in CLASSES:
    counts[cls]=download_class(cls,SEARCH_TERMS[cls])
    print(f'{cls:12s}: {counts[cls]} images downloaded')
    time.sleep(1)  # reduce search throttling

print('\nDataset folder:',DATA_DIR)
missing=[c for c,n in counts.items() if n<2]
if missing:
    raise RuntimeError('Too few images for: '+', '.join(missing)+'. Rerun this cell later or change the search term.')
print('Total images:',sum(counts.values()))


In [ ]:
import tensorflow as tf, numpy as np, pathlib, json, time
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
CLASSES = ['biscuit','chips','pen','scale','phone','calculator','headphone','laptop','books','pencils','erasers','keyboard','mouse','toothpaste','ball','toothbrush','wires']
IMG_SIZE=(160,160); BATCH=16; SEED=42
# Validate folders and collect images.
valid_ext={'.jpg','.jpeg','.png','.webp','.bmp'}
paths=[]; labels=[]
for idx, name in enumerate(CLASSES):
    folder=os.path.join(DATA_DIR,name)
    if not os.path.isdir(folder): print('MISSING FOLDER:', name); continue
    files_in=[p for p in pathlib.Path(folder).rglob('*') if p.suffix.lower() in valid_ext]
    print(f'{name:12s}: {len(files_in)} images')
    for p in files_in: paths.append(str(p)); labels.append(idx)
assert len(set(labels)) >= 2, 'Need images in at least two class folders.'
assert all(os.path.isdir(os.path.join(DATA_DIR,c)) for c in CLASSES), 'Please add all 17 folders before training.'
from collections import Counter
print('Total images:',len(paths),' | smallest class:',min(Counter(labels).values()))

## 2. Train the fast CNN

MobileNetV2 is pretrained on ImageNet; we reuse its visual features and train a small 17-class head. This is much faster than training a CNN from scratch. With only five images per class, the displayed validation score is a rough sanity check, **not** a reliable real-world accuracy estimate.

In [ ]:
# Stratified split only when each class has at least 2 images; keep a small validation set.
X_train, X_val, y_train, y_val = train_test_split(np.array(paths), np.array(labels), test_size=0.2, random_state=SEED, stratify=np.array(labels))
AUTOTUNE=tf.data.AUTOTUNE
def load_image(path, label):
    raw=tf.io.read_file(path)
    img=tf.image.decode_image(raw, channels=3, expand_animations=False)
    img=tf.image.resize(img, IMG_SIZE)
    img=preprocess_input(tf.cast(img,tf.float32))
    return img, tf.one_hot(label, len(CLASSES))
def make_ds(x,y,training=False):
    ds=tf.data.Dataset.from_tensor_slices((x,y))
    if training: ds=ds.shuffle(len(x),seed=SEED)
    ds=ds.map(load_image,num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
    return ds
train_ds=make_ds(X_train,y_train,True); val_ds=make_ds(X_val,y_val)
augment=tf.keras.Sequential([layers.RandomFlip('horizontal'),layers.RandomRotation(0.08),layers.RandomZoom(0.12),layers.RandomContrast(0.12)],name='augmentation')
base=MobileNetV2(input_shape=(*IMG_SIZE,3),include_top=False,weights='imagenet',pooling='avg')
base.trainable=False
inputs=layers.Input(shape=(*IMG_SIZE,3))
x=augment(inputs)
x=base(x,training=False)
x=layers.Dropout(0.25)(x)
outputs=layers.Dense(len(CLASSES),activation='softmax')(x)
cnn=models.Model(inputs,outputs)
cnn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),loss='categorical_crossentropy',metrics=['accuracy'])
history=cnn.fit(train_ds,validation_data=val_ds,epochs=8,verbose=1)
val_probs=cnn.predict(val_ds,verbose=0)
val_pred=val_probs.argmax(axis=1)
print('Validation accuracy (small split; not a reliable generalization estimate):', round(accuracy_score(y_val,val_pred),3))
print(classification_report(y_val,val_pred,labels=list(range(len(CLASSES))),target_names=CLASSES,zero_division=0))
cnn.save('/content/product_cnn_mobilenetv2.keras')
with open('/content/product_classes.json','w') as f: json.dump(CLASSES,f)
print('Saved CNN and class list.')

## 3. Load the live detector

YOLO-World proposes real bounding boxes; the trained CNN classifies each detected crop into your 17 custom categories. Inference is run on resized frames for speed. On a free CPU runtime it may not feel instant; GPU is recommended.

In [ ]:
from ultralytics import YOLO
import cv2, gradio as gr, PIL.Image as Image
detector=YOLO('yolov8s-worldv2.pt')
PROMPTS=['biscuit packet','chips packet','pen','ruler','mobile phone','calculator','headphones','laptop','book','pencils','eraser','computer keyboard','computer mouse','toothpaste tube','ball','toothbrush','wires or cables']
detector.set_classes(PROMPTS)
cnn=tf.keras.models.load_model('/content/product_cnn_mobilenetv2.keras')
print('Detector and CNN ready. Webcam inference will start in the next cell.')

In [ ]:
# Real-time Gradio webcam. Each incoming frame is processed; green boxes are detector-generated, not a fixed overlay.
def recognize_frame(frame):
    if frame is None: return None
    # Gradio webcam frames arrive as RGB arrays.
    frame=np.asarray(frame).astype(np.uint8)
    h,w=frame.shape[:2]
    scale=min(640/max(w,h),1.0)
    small=cv2.resize(frame,(int(w*scale),int(h*scale))) if scale<1 else frame
    result=detector.predict(small,conf=0.12,iou=0.45, imgsz=640, verbose=False)[0]
    output=frame.copy()
    if result.boxes is None or len(result.boxes)==0:
        cv2.putText(output,'No object detected - move closer / improve lighting',(12,28),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,180,0),2)
        return output
    sx=w/small.shape[1]; sy=h/small.shape[0]
    # Prepare all detected crops and classify them in one batch (faster than one model call per box).
    detections=[]; crops=[]
    for box in result.boxes:
        x1,y1,x2,y2=map(float,box.xyxy[0].cpu().numpy())
        x1,y1,x2,y2=int(x1*sx),int(y1*sy),int(x2*sx),int(y2*sy)
        x1=max(0,x1); y1=max(0,y1); x2=min(w,x2); y2=min(h,y2)
        if x2-x1<12 or y2-y1<12: continue
        crop=cv2.resize(frame[y1:y2,x1:x2],IMG_SIZE).astype(np.float32)
        crops.append(preprocess_input(crop))
        detections.append((x1,y1,x2,y2))
    if crops:
        probs_batch=cnn.predict(np.stack(crops),batch_size=16,verbose=0)
        for (x1,y1,x2,y2),probs in zip(detections,probs_batch):
            idx=int(np.argmax(probs)); confidence=float(probs[idx])
            label=CLASSES[idx] if confidence>=0.25 else 'unknown'
            text=f'{label} {confidence:.0%}'
            cv2.rectangle(output,(x1,y1),(x2,y2),(0,255,0),3)
            (tw,th),_=cv2.getTextSize(text,cv2.FONT_HERSHEY_SIMPLEX,0.65,2)
            top=max(0,y1-th-10)
            cv2.rectangle(output,(x1,top),(x1+tw+8,top+th+8),(0,255,0),-1)
            cv2.putText(output,text,(x1+4,top+th+2),cv2.FONT_HERSHEY_SIMPLEX,0.65,(0,0,0),2)
    return output

demo=gr.Interface(fn=recognize_frame,inputs=gr.Image(sources=['webcam'],streaming=True,type='numpy',label='Live camera'),outputs=gr.Image(type='numpy',label='Live detection with green boxes'),live=True,stream_every=0.25, title='Live Product Recognition — CNN + YOLO-World', description='Keep the product visible and well-lit. Frames update as you move or change the object. Detector may miss categories it was not trained to localize.')
demo.launch(share=True,debug=False)

## 4. Export trained files

Download the trained CNN and class list if you want to reuse them. The YOLO-World weights are downloaded automatically by Ultralytics when first used.


In [ ]:
from google.colab import files
files.download('/content/product_cnn_mobilenetv2.keras')
files.download('/content/product_classes.json')